# Pipeline for counting EFMs and calculating their thermodynamic properties

This notebook builds a COBRApy model of biochemical reactions from a custom Excel spreadsheet. It (1) checks stoichiometric balance of every reaction using the KEGG REST API, (2) runs Flux Balance Analysis (FBA) to compute maximum theoretical product yields from glucose, and (3) exports the final model to JSON and SBML formats.

## User Inputs

Edit **only the cell below**, then run all cells from top to bottom.

At a few points in the notebook you will encounter a box like this:

<div style="border-left:5px solid #2196F3; background:rgba(33,150,243,0.08); padding:8px 14px; margin:8px 0; border-radius:3px;">
<b>&#9998; TO DO:</b> Example action required.
</div>

These mark steps where you need to **check the output and possibly edit your Excel file** before continuing. Re-run all cells after making any changes.

| Variable | What to change |
|---|---|
| `EXCEL_FILE` | Name of your Excel file (must be in the same folder as this notebook) |
| `MODEL_NAME` | Name used for the output `.json` and `.xml` files |
| `SUBSTRATES` | Exchange reaction ID(s) for compounds the pathway **consumes** |
| `PRODUCTS` | Exchange reaction IDs for **all** secreted products (including energy currency) |
| `CARBON_PRODUCTS` | Subset of `PRODUCTS` to check maximum yields for (usually excludes ATP) |
| `ENERGY_PRODUCT` | The ATP (or equivalent) exchange ID — used to detect thermodynamically infeasible cycles |
| `REV_ALLOWED` | Metabolites that can be both taken up **and** secreted freely |
| `PSEUDO_METS` | Currency metabolites whose transport is **not** counted as an enzymatic step |


In [ ]:
# ============================================================
# USER INPUTS — Edit this cell, then run all cells
# ============================================================

# Excel file with your pathway (must be in the same folder as this notebook)
EXCEL_FILE = 'EFMexamples_8.xlsx'

# Name for your model and output files (no spaces)
MODEL_NAME = 'EFMexamples_8'

# -------------- below are already arranged in the spreadsheet ------------------
# # Exchange reaction ID for the substrate (compound the pathway consumes)
# SUBSTRATES = ['EXGLC']

# # All exchange reaction IDs that can be secreted as products
# PRODUCTS = ['EXATP', 'EXETOH', 'EXAC', 'EXFOR', 'EXLAC', 'EXCO2']

# # Products to check maximum yield for (exclude energy currency such as ATP)
# CARBON_PRODUCTS = ['EXETOH', 'EXAC', 'EXFOR', 'EXLAC', 'EXCO2']

# # Exchange reaction ID for the energy currency product (used to detect infeasible cycles)
# ENERGY_PRODUCT = 'EXATP'

# # Cofactor/currency exchanges that can be freely taken up or secreted
# REV_ALLOWED = ['EXADP', 'EXPi', 'EXH2O', 'EXH']

# # Currency metabolites whose transport is NOT counted as an enzymatic step
PSEUDO_METS = {'H2O', 'H2Oex', 'H', 'Hex', 'ATP', 'ATPex', 'ADP', 'ADPex', 'Pi', 'Piex'}


## Imports

If you are running in **Google Colab**, add a code cell at the top and run:

```python
!pip install cobra
!pip install efmtool
!pip install equilibrator-api
```

> **Note:** `equilibrator-api` downloads thermodynamic data on first use — this can take a few minutes depending on your internet connection.


In [2]:
import math
import re
import time
from collections import Counter
from functools import lru_cache

import cobra
import efmtool
import numpy as np
import pandas as pd
import requests

In [3]:
# Try to only run this once (only when you need it), it takes a while (depending on your wifi)
from equilibrator_api import ComponentContribution, Q_
cc = ComponentContribution()

c:\Users\mre283\AppData\Local\anaconda3\envs\education-pipeline\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Helper Functions

Two utility functions used throughout the notebook: `dict_to_kegg_reaction` converts a metabolite stoichiometry dictionary into a KEGG-formatted reaction string (needed for eQuilibrator thermodynamics lookups), and `clean_dataframe_whitespace` strips newline and tab characters from all string cells in a DataFrame to ensure consistent parsing.

In [6]:
# Helper function for eQuilibrator
def dict_to_kegg_reaction(met_dict):
    reactants = []
    products = []

    for met, coeff in met_dict.items():
        if coeff < 0:
            term = f"{-coeff:g} kegg:{met}" if abs(coeff) != 1 else f"kegg:{met}"
            reactants.append(term)
        elif coeff > 0:
            term = f"{coeff:g} kegg:{met}" if abs(coeff) != 1 else f"kegg:{met}"
            products.append(term)

    lhs = ' + '.join(reactants)
    rhs = ' + '.join(products)
    return f"{lhs} = {rhs}"

In [7]:
def clean_dataframe_whitespace(df, inplace=False):
    """
    Remove all newline '\n' and tab '\t' characters from every string cell
    in a pandas DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
        The dataframe to clean.
    inplace : bool, optional
        If True, modify the dataframe in place. Default is False.

    Returns
    -------
    pandas.DataFrame
        The cleaned dataframe (unless inplace=True).
    """

    target = df if inplace else df.copy()

    target = target.applymap(
        lambda x: x.replace('\n', '').replace('\t', '') if isinstance(x, str) else x
    )

    return target

## Load Reaction and Metabolite Data

Read the metabolite table and reaction list from the Excel workbook (`Example1_EMPglycolysis.xlsx`). Whitespace is cleaned from both DataFrames before further processing.

In [8]:
df_metabolites = pd.read_excel(EXCEL_FILE, sheet_name='Metabolites')
df_reactions = pd.read_excel(EXCEL_FILE, sheet_name='Reactions')

df_metabolites = clean_dataframe_whitespace(df_metabolites)
df_reactions = clean_dataframe_whitespace(df_reactions)


C:\Users\mre283\AppData\Local\Temp\ipykernel_9488\3564247440.py:21: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  target = target.applymap(


## Stoichiometric Balance Checking — Function Definitions

Defines the full pipeline for checking whether each reaction is atom- and charge-balanced:

- **KEGG API fetching** (`get_compound_info`): retrieves molecular formula and charge for a KEGG compound ID, with caching.
- **Equation parsing** (`parse_equation`, `normalize_arrow`): splits a reaction string into substrate and product lists.
- **Formula parsing** (`parse_formula`): converts a KEGG formula string (e.g. `C6H12O6`) into an element count dictionary; handles hydrates and unusual characters.
- **ID mapping** (`map_custom_to_kegg_equation`): translates the custom abbreviations used in the Excel sheet to KEGG compound IDs.
- **Balance check** (`check_balance`): computes the atom and charge delta across a reaction; returns a summary of any imbalances.
- **Pipeline functions** (`analyze_custom_equations`, `analyze_stoichiometry_matrix`): orchestrate the above steps for a list of reactions and return a results DataFrame.

In [9]:

# -------- ELEMENT SYMBOL WHITELIST (up to Og) --------
ELEMENTS = {
    "H","He","Li","Be","B","C","N","O","F","Ne","Na","Mg","Al","Si","P","S","Cl","Ar",
    "K","Ca","Sc","Ti","V","Cr","Mn","Fe","Co","Ni","Cu","Zn","Ga","Ge","As","Se","Br","Kr",
    "Rb","Sr","Y","Zr","Nb","Mo","Tc","Ru","Rh","Pd","Ag","Cd","In","Sn","Sb","Te","I","Xe",
    "Cs","Ba","La","Ce","Pr","Nd","Pm","Sm","Eu","Gd","Tb","Dy","Ho","Er","Tm","Yb","Lu",
    "Hf","Ta","W","Re","Os","Ir","Pt","Au","Hg","Tl","Pb","Bi","Po","At","Rn",
    "Fr","Ra","Ac","Th","Pa","U","Np","Pu","Am","Cm","Bk","Cf","Es","Fm","Md","No","Lr",
    "Rf","Db","Sg","Bh","Hs","Mt","Ds","Rg","Cn","Nh","Fl","Mc","Lv","Ts","Og"
}

# -------- KEGG ID pattern (C/D/G + 5 digits) --------
_KEGG_ID_RE = re.compile(r"^[CDG]\d{5}$")

# -------- KEGG API (robust parsing + caching) --------
@lru_cache(maxsize=10000)
def get_compound_info(cid):
    """
    Return (formula_str, charge_int) from KEGG REST for a Cxxxxx id.
    Robust to variable spacing; ignores non-matching lines.
    """
    url = f"http://rest.kegg.jp/get/{cid}"
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    formula, charge = None, 0
    for line in r.text.splitlines():
        # FORMULA <spaces> <formula>
        m = re.match(r"^FORMULA\s+(.+)$", line)
        if m:
            formula = m.group(1).strip()
            continue
        # CHARGE <spaces> <int>
        m = re.match(r"^CHARGE\s+(-?\d+)$", line)
        if m:
            try:
                charge = int(m.group(1))
            except ValueError:
                pass
    return formula, charge

# -------- Equation parsing (robust to common arrows) --------
def normalize_arrow(eq):
    # normalize a few common arrows to "<=>"
    eq = re.sub(r"\s*(<[-=]+>|<=>|<->|⇌|↔|→|->|=>)\s*", " <=> ", eq)
    return eq

def parse_equation(equation):
    """
    Parse a reaction equation into substrates and products.
    Supports integer and fractional stoichiometric coefficients.
    Returns [(coeff, compound_id), ...] for each side.
    """
    equation = normalize_arrow(equation)
    if "<=>" not in equation:
        raise ValueError(f"No recognized arrow in equation: {equation}")
    left, right = equation.split("<=>")

    def parse_side(side):
        parts = [p.strip() for p in side.strip().split("+") if p.strip()]
        parsed = []
        for p in parts:
            # allow int or float coefficient
            m = re.match(r"^\s*([\d\.]+)\s+(\S+)\s*$", p)
            if m:
                coeff = float(m.group(1))
                cid = m.group(2)
            else:
                coeff, cid = 1.0, p.strip()
            parsed.append((coeff, cid))
        return parsed

    return parse_side(left), parse_side(right)

# -------- Formula parsing (only real elements, supports hydrates) --------
def parse_formula(formula):
    """
    Parse a chemical formula string into element counts.
    Supports dot/center-dot separated parts (e.g., 'C10H16N5O13P3·H2O').
    Ignores any stray characters outside [A-Za-z0-9·.].
    """
    if not formula:
        return Counter()

    counts = Counter()
    # split hydrate parts: dot or middle dot or whitespace clusters
    parts = re.split(r"[·\.]\s*|\s{2,}", formula.strip())
    for part in parts:
        if not part:
            continue
        # strip any non-alnum that might sneak in
        part = re.sub(r"[^A-Za-z0-9]", "", part)
        # scan symbol-by-symbol (1 or 2-letter symbols only)
        i = 0
        while i < len(part):
            if i+1 < len(part) and part[i:i+2] in ELEMENTS:
                sym = part[i:i+2]; i += 2
            elif part[i] in [s[:1] for s in ELEMENTS]:
                # take single letter; verify it's a valid 1-letter symbol
                sym1 = part[i]; i += 1
                # map to full symbol if exists (H, B, C, N, O, F, P, S, K, V, Y, I, W, U)
                if sym1 in ELEMENTS:
                    sym = sym1
                else:
                    # not a valid element (e.g., 'R' from 'FORMULA'); skip
                    continue
            else:
                # unknown letter chunk; skip it (guards against stray text)
                i += 1
                continue

            # optional number following the symbol
            j = i
            while j < len(part) and part[j].isdigit():
                j += 1
            num = int(part[i:j]) if j > i else 1
            counts[sym] += num
            i = j
    return counts

def multiply_formula(counts, coeff):
    return Counter({atom: n * coeff for atom, n in counts.items()})

# -------- Clean mapping: custom IDs -> KEGG IDs by token --------
def map_custom_to_kegg_equation(equation, mapping_df):
    custom2kegg = dict(zip(mapping_df["ID"], mapping_df["KEGG ID"]))
    subs, prods = parse_equation(equation)

    def map_side(pairs):
        mapped = []
        for coeff, cid in pairs:
            kid = custom2kegg.get(cid, cid)
            if pd.isna(kid) or kid is None:
                kid = cid  # leave unmapped if no KEGG id
            mapped.append((coeff, str(kid)))
        return mapped

    subs_k = map_side(subs)
    prods_k = map_side(prods)

    def side_to_str(pairs):
        parts = []
        for coeff, cid in pairs:
            if abs(coeff - 1.0) < 1e-12:  # display clean if coefficient is ~1
                parts.append(cid)
            else:
                parts.append(f"{coeff:g} {cid}")  # keep decimals clean
        return " + ".join(parts) if parts else "0"

    eq_kegg = f"{side_to_str(subs_k)} <=> {side_to_str(prods_k)}"
    return eq_kegg, subs_k, prods_k

# -------- Balance check with signed atom deltas --------
def check_balance(substrates, products, formula_map, charge_map, tol=1e-6):
    left_atoms, right_atoms = Counter(), Counter()
    left_charge, right_charge = 0.0, 0.0

    for coeff, cid in substrates:
        f = formula_map.get(cid)
        if f:
            left_atoms += multiply_formula(parse_formula(f), coeff)
        left_charge += charge_map.get(cid, 0) * coeff

    for coeff, cid in products:
        f = formula_map.get(cid)
        if f:
            right_atoms += multiply_formula(parse_formula(f), coeff)
        right_charge += charge_map.get(cid, 0) * coeff

    # compare atoms with tolerance
    all_atoms = set(left_atoms) | set(right_atoms)
    delta = {
        a: right_atoms.get(a, 0.0) - left_atoms.get(a, 0.0)
        for a in all_atoms
    }

    atoms_bal = all(abs(d) < tol for d in delta.values())
    charge_bal = abs(left_charge - right_charge) < tol

    # compact summary string for imbalances above tolerance
    imbalance_summary = ", ".join(
        [f"{'+' if d > 0 else ''}{d:g} {a}" for a, d in sorted(delta.items()) if abs(d) >= tol]
    ) or None

    return atoms_bal, charge_bal, left_atoms, right_atoms, left_charge, right_charge, delta, imbalance_summary

# -------- Build local formula fallback from df_metabolites --------
def _build_local_formula_map(mapping_df):
    """
    Return a dict {custom_ID: formula_str} for rows where KEGG ID is empty
    but a 'Chemical formula' is provided.
    """
    local = {}
    for _, row in mapping_df.iterrows():
        kegg = row.get("KEGG ID")
        if pd.isna(kegg) or not str(kegg).strip():
            formula = row.get("Chemical formula")
            if pd.notna(formula) and str(formula).strip():
                local[row["ID"]] = str(formula).strip()
    return local

# -------- Pipeline: custom eq -> KEGG eq -> balance --------
def analyze_custom_equations(equations, mapping_df):
    # Formulas for metabolites that have no KEGG ID
    local_formula = _build_local_formula_map(mapping_df)

    rows = []
    for eq in equations:
        try:
            eq_kegg, subs_k, prods_k = map_custom_to_kegg_equation(eq, mapping_df)
        except Exception as e:
            rows.append({
                "equation_custom": eq,
                "equation_kegg": None,
                "atom_balanced": None,
                "charge_balanced": None,
                "left_charge": None,
                "right_charge": None,
                "atom_delta": None,
                "atom_imbalance": None,
                "notes": f"Parse error: {e}",
            })
            continue

        # collect all compound IDs in the mapped equation
        cids = [cid for _, cid in subs_k + prods_k]

        # build formula/charge maps: use local Chemical formula when no KEGG ID
        uniq = list(dict.fromkeys(cids))  # preserve order, unique
        formula_map, charge_map = {}, {}
        missing_cids = []
        for cid in uniq:
            if cid in local_formula:
                formula_map[cid] = local_formula[cid]
                charge_map[cid] = 0
            elif _KEGG_ID_RE.match(cid):
                info = get_compound_info(cid)
                formula_map[cid] = info[0]
                charge_map[cid] = info[1]
            else:
                missing_cids.append(cid)
        if missing_cids:
            print("Warning: skipping reaction " + repr(eq) +
                  " — no KEGG ID or Chemical formula for: " + str(missing_cids))
            rows.append({
                "equation_custom": eq,
                "equation_kegg": eq_kegg,
                "atom_balanced": None,
                "charge_balanced": None,
                "left_charge": None,
                "right_charge": None,
                "atom_delta": None,
                "atom_imbalance": None,
                "notes": "Skipped: missing formula for " + str(missing_cids),
            })
            continue

        atoms_bal, charge_bal, left_atoms, right_atoms, left_charge, right_charge, delta, imbalance_summary = \
            check_balance(subs_k, prods_k, formula_map, charge_map)

        notes = []
        if not atoms_bal:
            notes.append("Atoms not balanced")
        if not charge_bal:
            notes.append("Charge not balanced")
        if not notes:
            notes.append("OK")

        rows.append({
            "equation_custom": eq,
            "equation_kegg": eq_kegg,
            "atom_balanced": atoms_bal,
            "charge_balanced": charge_bal,
            "left_charge": left_charge,
            "right_charge": right_charge,
            "atom_delta": delta,                      # dict, e.g. {'H': -1, 'P': +1}
            "atom_imbalance": imbalance_summary,      # string, e.g. "+1 P, -1 H"
            "notes": "; ".join(notes),
        })

    return pd.DataFrame(rows)

def analyze_stoichiometry_matrix(
    stoich_df,
    mapping_df,
    formula_map=None,
    charge_map=None,
    tol=1e-6,
    only_unbalanced=False,
):
    """
    Analyze reactions from a stoichiometric matrix format.

    Parameters
    ----------
    stoich_df : pd.DataFrame
        Rows = metabolites (index like 'EXH2O', 'EXATP', etc., abbrev = index[2:]).
        Columns = reactions, values = stoichiometric coefficients.
        Negative = substrate, Positive = product.

    mapping_df : pd.DataFrame
        Must contain columns ["ID", "KEGG ID"]. Optionally "Chemical formula"
        as fallback for rows where "KEGG ID" is empty.

    formula_map, charge_map : dict, optional
        Precomputed maps {kegg_id: formula_str} and {kegg_id: charge_int}.
        If not provided, compound info will be fetched via KEGG (with caching),
        falling back to "Chemical formula" for IDs without a KEGG mapping.

    tol : float, default 1e-6
        Tolerance for atom/charge balance checks.

    only_unbalanced : bool, default False
        If True, only return reactions that are not atom- and/or charge-balanced.

    Returns
    -------
    pd.DataFrame with one row per reaction, including balance check results.
    """
    rows = []
    abbrev2kegg = dict(zip(mapping_df["ID"], mapping_df["KEGG ID"]))
    # Formulas for metabolites that have no KEGG ID
    local_formula = _build_local_formula_map(mapping_df)

    for rxn in stoich_df.columns:
        coeffs = stoich_df[rxn].dropna()
        substrates, products = [], []

        for met_full, coeff in coeffs.items():
            if abs(coeff) < tol:
                continue
            # abbreviation from index[2:]
            abbrev = met_full[2:]
            kegg_id = abbrev2kegg.get(abbrev, abbrev)
            if pd.isna(kegg_id) or kegg_id is None:
                kegg_id = abbrev

            if coeff < 0:
                substrates.append((-coeff, str(kegg_id)))
            else:
                products.append((coeff, str(kegg_id)))

        # Build KEGG-style equation string
        def side_to_str(pairs):
            parts = []
            for coeff, cid in pairs:
                if abs(coeff - 1.0) < tol:
                    parts.append(cid)
                else:
                    parts.append(f"{coeff:g} {cid}")
            return " + ".join(parts) if parts else "0"

        eq_kegg = f"{side_to_str(substrates)} <=> {side_to_str(products)}"

        # Build maps if not precomputed
        if formula_map is None or charge_map is None:
            cids = [cid for _, cid in substrates + products]
            uniq = list(dict.fromkeys(cids))
            f_map, c_map = {}, {}
            missing_cids = []
            for cid in uniq:
                if cid in local_formula:
                    f_map[cid] = local_formula[cid]
                    c_map[cid] = 0
                elif _KEGG_ID_RE.match(cid):
                    info = get_compound_info(cid)
                    f_map[cid] = info[0]
                    c_map[cid] = info[1]
                else:
                    missing_cids.append(cid)
            if missing_cids:
                print("Warning: skipping reaction " + repr(rxn) +
                      " — no KEGG ID or Chemical formula for: " + str(missing_cids))
                rows.append({
                    "reaction_id": rxn,
                    "equation_kegg": eq_kegg,
                    "atom_balanced": None,
                    "charge_balanced": None,
                    "left_charge": None,
                    "right_charge": None,
                    "atom_delta": None,
                    "atom_imbalance": None,
                    "notes": "Skipped: missing formula for " + str(missing_cids),
                })
                continue
        else:
            f_map, c_map = formula_map, charge_map

        # Check balance
        atoms_bal, charge_bal, left_atoms, right_atoms, left_charge, right_charge, delta, imbalance_summary = \
            check_balance(substrates, products, f_map, c_map, tol=tol)

        # Skip balanced reactions if requested
        if only_unbalanced and atoms_bal and charge_bal:
            continue

        notes = []
        if not atoms_bal:
            notes.append("Atoms not balanced")
        if not charge_bal:
            notes.append("Charge not balanced")
        if not notes:
            notes.append("OK")

        rows.append({
            "reaction_id": rxn,
            "equation_kegg": eq_kegg,
            "atom_balanced": atoms_bal,
            "charge_balanced": charge_bal,
            "left_charge": left_charge,
            "right_charge": right_charge,
            "atom_delta": delta,
            "atom_imbalance": imbalance_summary,
            "notes": "; ".join(notes),
        })

    return pd.DataFrame(rows)


## Run Stoichiometric Balance Check

Apply the balance-checking pipeline to every reaction in the loaded dataset. Reaction equations are first translated from custom abbreviations to KEGG compound IDs, then checked for atom and charge balance. Results (including any imbalance details) are stored in `df_results`.

<div style="border-left:5px solid #2196F3; background:rgba(33,150,243,0.08); padding:10px 14px; margin:12px 0; border-radius:3px;">
<b>&#9998; TO DO:</b> Inspect <code>df_results</code> below. If any <b>non-exchange</b> reaction is imbalanced, edit your Excel file to fix it, then re-run from the top. H2O and H+; may be added to either side of a reaction to balance atoms and charge as indicated by the imbalance shown. In the column "Atom imbalance", there is an indication of what elements are missing: -1 H means that a H+ should be added to the right side or removed from the left side, while +1 H means that a H+ should be added to the left side or removed from the right side. Exchange reactions are by definition imbalanced, so those will appear as imbalanced and no action is required for those. 
</div>


In [10]:
# # equations written in your custom Abbreviations
reactions = list(df_reactions.loc[:, 'Reaction stoichiometry'])

# # mapping table (df_metabolites) should have: ["KEGG ID", "ID"]
df_results = analyze_custom_equations(reactions, df_metabolites)

In [11]:
df_results

,equation_custom,equation_kegg,atom_balanced,charge_balanced,left_charge,right_charge,atom_delta,atom_imbalance,notes
0,GLCex + PEP <=> G6P + PYR,C00031 + C00074 <=> C00092 + C00022,True,True,0.0,0.0,"{'C': 0.0, 'O': 0.0, 'H': 0.0, 'P': 0.0}",None,OK
1,GLC + ATP <=> G6P + ADP,C00031 + C00002 <=> C00092 + C00008,True,True,0.0,0.0,"{'C': 0.0, 'N': 0.0, 'O': 0.0, 'H': 0.0, 'P': ...",None,OK
2,G6P <=> F6P,C00092 <=> C00085,True,True,0.0,0.0,"{'C': 0.0, 'O': 0.0, 'H': 0.0, 'P': 0.0}",None,OK
3,F6P + ATP <=> FBP + ADP,C00085 + C00002 <=> C00354 + C00008,True,True,0.0,0.0,"{'C': 0.0, 'N': 0.0, 'O': 0.0, 'H': 0.0, 'P': ...",None,OK
4,FBP <=> G3P + DHAP,C00354 <=> C00118 + C00111,True,True,0.0,0.0,"{'C': 0.0, 'O': 0.0, 'H': 0.0, 'P': 0.0}",None,OK
...,...,...,...,...,...,...,...,...,...
78,Hex <=>,C00080 <=> 0,False,True,0.0,0.0,{'H': -1.0},-1 H,Atoms not balanced
79,H2 <=> H2ex,C00282 <=> C00282,True,True,0.0,0.0,{'H': 0.0},None,OK
80,H2ex <=>,C00282 <=> 0,False,True,0.0,0.0,{'H': -2.0},-2 H,Atoms not balanced
81,CH4 <=> CH4ex,C01438 <=> C01438,True,True,0.0,0.0,"{'C': 0.0, 'H': 0.0}",None,OK


## For loop

In this case, because we are calculating whether 8 pathways are each a single EFM, we do this in a for loop. We still build the model with all reactions, but then set bounds and remove reactions for the confirmation of each EFM. We set the reversibility bounds for each EFM separately, so to start we just set them all to irreversible. 

In [12]:
model = cobra.Model(name=MODEL_NAME)

cobra_mets = []
for i in df_metabolites.index:
  cobra_mets.append(cobra.Metabolite(id=df_metabolites.loc[i, 'ID'], name=df_metabolites.loc[i, 'Name'], compartment='c'))

model.add_metabolites(cobra_mets)

cobra_reacs = []

for i in df_reactions.index:

  r = cobra.Reaction(id=df_reactions.loc[i, 'ID'], name=df_reactions.loc[i, 'Name'], lower_bound=0, upper_bound=1000)

  model.add_reactions([r])

  r.reaction = df_reactions.loc[i, 'Reaction stoichiometry']



Set parameter Username
Set parameter LicenseID to value 2768361
Academic license - for non-commercial use only - expires 2027-01-21


## Helpers for printing and dG calculation

In [13]:
def print_overall_reaction(df_normalized):
    ex_rows = df_normalized[df_normalized.index.str.startswith('EX')]

    reactants = []
    products = []

    for rxn_id, fluxes in ex_rows.iterrows():
        nonzero_fluxes = fluxes[fluxes != 0]
        if nonzero_fluxes.empty:
            continue
        coeff = nonzero_fluxes.iloc[0]

        met_name = rxn_id[2:]
        if coeff < 0:
            reactants.append(f"{abs(coeff):.2f} {met_name}" if abs(coeff) != 1 else met_name)
        else:
            products.append(f"{coeff:.2f} {met_name}" if coeff != 1 else met_name)

    lhs = " + ".join(reactants) if reactants else "∅"
    rhs = " + ".join(products) if products else "∅"

    print(f"  Overall reaction: {lhs} → {rhs}")


def calculate_dG(df_normalized, df_metabolites, cc):
    """
    Calculate dG0prime and dGm for each EFM in df_normalized.
    Adds dG0prime value/error, dGm value/error, and per-reaction normalized rows.
    
    Args:
        df_normalized:  normalized EFM DataFrame (rows = reaction IDs, cols = EFMs)
        df_metabolites: DataFrame with 'ID' and 'KEGG ID' columns
        cc:             equilibrator_api ComponentContribution instance
    
    Returns:
        df_normalized with dG rows appended
    """
    metabolite_map = dict(zip(df_metabolites['ID'], df_metabolites['KEGG ID']))

    df_normalized.loc['dG0prime value'] = pd.Series(dtype=object)
    df_normalized.loc['dG0prime error'] = pd.Series(dtype=object)
    df_normalized.loc['dGm value']      = pd.Series(dtype=object)
    df_normalized.loc['dGm error']      = pd.Series(dtype=object)

    for reac in df_normalized.columns:
        series = df_normalized.loc[df_normalized.index.str.startswith('EX'), reac]
        reac_dict = series[series != 0].to_dict()

        kegg_dict = {}
        for k, v in reac_dict.items():
            metab = k[2:]
            kegg_code = metabolite_map[metab]
            kegg_dict[kegg_code] = v

        kegg_string = dict_to_kegg_reaction(kegg_dict)
        overall_reaction = cc.parse_reaction_formula(kegg_string)

        dG0_prime = cc.standard_dg_prime(overall_reaction)
        dGm       = cc.physiological_dg_prime(overall_reaction)

        df_normalized.loc['dG0prime value', reac] = dG0_prime.value.magnitude
        df_normalized.loc['dG0prime error', reac] = dG0_prime.error.magnitude
        df_normalized.loc['dGm value', reac]      = dGm.value.magnitude
        df_normalized.loc['dGm error', reac]      = dGm.error.magnitude

    df_normalized.loc['dG0prime value/nReactions'] = (
        df_normalized.loc['dG0prime value'] / df_normalized.loc['n_metabolic_reactions']
    )
    df_normalized.loc['dGm value/nReactions'] = (
        df_normalized.loc['dGm value'] / df_normalized.loc['n_metabolic_reactions']
    )

    df_normalized.loc['dG0prime value/nFlux'] = (
        df_normalized.loc['dG0prime value'] / df_normalized.loc['total_reaction_events']
    )
    df_normalized.loc['dGm value/nFlux'] = (
         df_normalized.loc['dGm value'] / df_normalized.loc['total_reaction_events']
    )

    return df_normalized

## Calculate results

In [14]:
results = {}

# we have 8 pathways
for i in range(1,9):
    print(f"\nProcessing pathway {i} \n ----------------------------------------")

    model_pathway = model.copy()

    pathway_col = df_reactions.loc[:, f'Pathway {i}']
    for j in range(len(pathway_col)):
        if pathway_col[j] == -1:
            r = model_pathway.reactions.get_by_id(df_reactions.loc[j, 'ID'])
            r.bounds = -1000, 1000
        elif pathway_col[j] == 0:
            r = model_pathway.reactions.get_by_id(df_reactions.loc[j, 'ID'])
            model_pathway.remove_reactions([r])
        elif pathway_col[j] == 1:
            r = model_pathway.reactions.get_by_id(df_reactions.loc[j, 'ID'])
            r.bounds = 0, 1000

    S = cobra.util.array.create_stoichiometric_matrix(model_pathway)

    S_nonzero = S[~np.all(S == 0, axis=1)]
    metabolite_ids = np.array([m.id for m in model_pathway.metabolites])[~np.all(S == 0, axis=1)]

    S = S_nonzero

    reversibility_array = []
    for r in model_pathway.reactions:

        if r.lower_bound < 0:
            reversibility_array.append(1)
        else:
            reversibility_array.append(0)

    print(f"Shape of stoichiometric matrix: {S.shape}")

    start = time.time()

    efms = efmtool.calculate_efms(S, reversibility_array, [r.id for r in model_pathway.reactions], metabolite_ids)
    end = time.time()

    print(f'EFM calculation took {(end-start):.2f} s')
    print(f'Number of EFMs: {efms.shape[1]}')

    df_efms = pd.DataFrame(efms, index=[r.id for r in model_pathway.reactions], columns=[f'EFM_{i+1}' for i in range(efms.shape[1])])
    

    if i == 5:
        norm_flux = 'EXGLC'
    else:
        norm_flux = 'EXATP'

    
    denominator = np.abs(df_efms.loc[norm_flux].values)
    nonzero_mask = denominator != 0

    df_normalized = df_efms.copy()
    df_normalized.loc[:, nonzero_mask] = df_efms.loc[:, nonzero_mask] / denominator[nonzero_mask]

    print_overall_reaction(df_normalized)
    
    # Non-EX reactions whose entire metabolite set is within PSEUDO_METS
    pseudo_transport_rxns = {
        r.id for r in model.reactions
        if not r.id.startswith('EX')
        and set(m.id for m in r.metabolites).issubset(PSEUDO_METS)
    }

    # Rows to keep: not an exchange, not a pseudo-transport, not a thermodynamic summary row
    thermo_rows = {'dG0prime value', 'dG0prime error', 'dGm value', 'dGm error'}
    metabolic_only = df_normalized[
        ~df_normalized.index.str.startswith('EX') &
        ~df_normalized.index.isin(pseudo_transport_rxns) &
        ~df_normalized.index.isin(thermo_rows)
    ]



    # Count reactions with nonzero flux per EFM
    rxn_counts = (metabolic_only != 0).sum().rename('active_metabolic_reactions')
    event_counts = np.abs(metabolic_only).sum().rename('total_reaction_events')

    print(f"Number of active metabolic reactions per EFM: {rxn_counts.iloc[0]}")
    print(f"Total reaction events per EFM: {event_counts.iloc[0]}")
    df_normalized.loc['n_metabolic_reactions'] = pd.Series(dtype=object)
    df_normalized.loc['total_reaction_events'] = pd.Series(dtype=object)
    for efm in df_normalized.columns:
        df_normalized.loc['n_metabolic_reactions', efm] = rxn_counts[efm]
        df_normalized.loc['total_reaction_events', efm] = event_counts[efm]

    df_normalized = calculate_dG(df_normalized, df_metabolites, cc)
    results[f'Pathway {i}'] = df_normalized
    print(f'dGm value: {df_normalized.loc["dGm value"].iloc[0]:.2f} kJ/mol ATP (or glc for pathway 5)')
    print(f'dGm value/nReactions: {df_normalized.loc["dGm value/nReactions"].iloc[0]:.2f} kJ/mol ATP (or glc for pathway 5) per active metabolic reaction')
    print(f'dGm value/nFlux: {df_normalized.loc["dGm value/nFlux"].iloc[0]:.2f} kJ/reaction event')



Processing pathway 1 
 ----------------------------------------
Read LP format model from file C:\Users\mre283\AppData\Local\Temp\tmpayydp90p.lp
Reading time = 0.02 seconds
: 79 rows, 166 columns, 572 nonzeros


Shape of stoichiometric matrix: (33, 32)
EFM calculation took 5.46 s
Number of EFMs: 1
  Overall reaction: 0.33 GLC + ADP + Pi → 0.33 ETOH + 0.33 AC + 0.67 FOR + ATP + 0.67 H2O
Number of active metabolic reactions per EFM: 18
Total reaction events per EFM: 8.0
dGm value: -44.87 kJ/mol ATP (or glc for pathway 5)
dGm value/nReactions: -2.49 kJ/mol ATP (or glc for pathway 5) per active metabolic reaction
dGm value/nFlux: -5.61 kJ/reaction event

Processing pathway 2 
 ----------------------------------------
Read LP format model from file C:\Users\mre283\AppData\Local\Temp\tmpn9cckbfx.lp
Reading time = 0.02 seconds
: 79 rows, 166 columns, 572 nonzeros
Shape of stoichiometric matrix: (25, 24)
EFM calculation took 1.67 s
Number of EFMs: 1
  Overall reaction: 0.50 GLC + ADP + Pi → LAC + ATP + H2O
Number of active metabolic reactions per EFM: 12
Total reaction events per EFM: 9.0
dGm value: -58.22 kJ/mol ATP (or glc for pathway 5)
dGm value/nReactions: -4.85 kJ/mol ATP (or glc for pathway 5) 

In [20]:
results['Pathway 1']

,EFM_1
PTS,1.0
PGI,1.0
PFK,1.0
ALDO,1.0
TPI,1.0
GAPDH,2.0
PGK,2.0
PGM,2.0
ENO,2.0
PYK,1.0


In [13]:
results['Pathway 2']

,EFM_1
PTS,1.0
PGI,1.0
PFK,1.0
ALDO,1.0
TPI,1.0
GAPDH,2.0
PGK,2.0
PGM,2.0
ENO,2.0
PYK,1.0


In [14]:
results['Pathway 3']

,EFM_1
PTS,1.0
GAPDH,1.0
PGK,1.0
PGM,1.0
ENO,1.0
PYK,0.0
LDH,2.0
G6PDH,1.0
PGL,1.0
EDD,1.0


In [23]:
results['Pathway 4']

,EFM_1
PTS,1.0
PFK,2.0
ALDO,2.0
TPI,2.0
GAPDH,5.0
PGK,5.0
PGM,5.0
ENO,5.0
PYK,4.0
PFL,5.0


In [16]:
results['Pathway 5']

,EFM_1
PTS,1.0
PGM,2.0
ENO,2.0
PYK,1.0
LDH,2.0
G6PDH,1.0
PGL,1.0
GND,1.0
PRUK,1.0
RUBPK,1.0


In [28]:
results['Pathway 6']

,EFM_1
PTA,4.000000
ACK,4.000000
FDH,4.000000
FTHFS,4.000000
MTHFC,4.000000
MTHFD,4.000000
MTHFR,4.000000
CODH,4.000000
ACS,4.000000
HYD_NAD,8.000000


In [18]:
results['Pathway 7']

,EFM_1
FMFDH,2.0
FMFFT,2.0
MLHCH,2.0
MLHDH,2.0
MLHR,2.0
MHCMT,2.0
MCMR,2.0
CCFH2OR,2.0
HYDF420,4.0
ATPASE,1.0


In [19]:
results['Pathway 8']

,EFM_1
PTA,-2.0
ACK,-2.0
CODH,-2.0
MHCMT,2.0
MCMR,2.0
ACSH4MPT,-2.0
ECH,2.0
CCFOR,2.0
ATPASE,3.0
TAC,-2.0
